# hyp3-isce3 — custom-parameter NISAR GUNW demo (`uwg_demo`)

**Thesis:** the operational NISAR L2 GUNWs are good. This repo's value is **parameter control** —
you can crop a pair, retune the runconfig, and reprocess for regimes/products the fixed 80 m
operational pipeline doesn't serve.

Three walkthroughs:
1. **What knobs are available** vs the standard GUNW (Cerro Prieto, cached)
2. **Mexico City** — more pairs: naive cumulative sum **vs MintPy** time-series inversion
3. **Wet snow** — a temporal **skip pair** that bridges over the wet acquisition (Susanville)

Environment: conda env `hyp3-isce3`. Demo data lives under `nisar_demo_subsets/`
(cropped RSLC caches + a few generated GUNWs). Cells that stream operational GUNWs need
Earthdata creds in `~/.netrc`.

In [ ]:
import glob, warnings
import numpy as np
import matplotlib.pyplot as plt
import h5py, earthaccess
from pyproj import Transformer
warnings.filterwarnings('ignore')
%matplotlib inline

earthaccess.login(strategy='netrc')

BASE = '/Users/zmhoppinen/Documents/nisar_demo_subsets'   # demo data root
LAM = 0.2379                                              # NISAR L-band wavelength [m]; 1 fringe = 11.9 cm LOS
GA = '/science/LSAR/GUNW/grids/frequencyA'                # GUNW frequency-A grids group


def read_gunw(path_or_url, aoi, layer='wrapped', stream=False):
    """Clip one GUNW layer to a lon/lat AOI. layer in {wrapped, coh, phase, cc}.

    Works on a local file path or (stream=True) an operational product URL.
    Returns (array, extent_km) where extent is [xmin,xmax,ymin,ymax] in UTM km.
    """
    src = earthaccess.open([path_or_url])[0] if stream else path_or_url
    f = h5py.File(src, 'r')
    if layer == 'wrapped':
        base, ds = f'{GA}/wrappedInterferogram', 'wrappedInterferogram'
    else:
        base = f'{GA}/unwrappedInterferogram'
        ds = {'coh': 'coherenceMagnitude', 'phase': 'unwrappedPhase', 'cc': 'connectedComponents'}[layer]
    epsg = int(f[f'{base}/HH/projection'].attrs['epsg_code'])
    tr = Transformer.from_crs(4326, epsg, always_xy=True)
    cs = [tr.transform(lo, la) for lo in (aoi[0], aoi[2]) for la in (aoi[1], aoi[3])]
    xs, ys = zip(*cs)
    x = f[f'{base}/xCoordinates'][:]; y = f[f'{base}/yCoordinates'][:]
    i = np.where((x >= min(xs)) & (x <= max(xs)))[0]; j = np.where((y >= min(ys)) & (y <= max(ys)))[0]
    arr = f[f'{base}/HH/{ds}'][j.min():j.max() + 1, i.min():i.max() + 1]
    xa, ya = x[i.min():i.max() + 1], y[j.min():j.max() + 1]
    f.close()
    ext = [xa.min() / 1000, xa.max() / 1000, ya.min() / 1000, ya.max() / 1000]
    return np.asarray(arr), ext


def wphase(cplx):
    """Wrapped phase [rad] from the (normalized) complex wrapped interferogram."""
    return np.angle(cplx)


def to_los_cm(unw_phase):
    """Unwrapped phase -> LOS displacement [cm]; masks the unwrap-fill zeros."""
    los = -unw_phase * LAM / (4 * np.pi) * 100.0
    los[unw_phase == 0] = np.nan
    return los - np.nanmedian(los)


---
## 1. What knobs are available (vs the standard GUNW)

The operational GUNW is produced with a **fixed** runconfig (5×6 crossmul looks, 80 m geocode
posting, tropo+iono on, full coregistration). This repo takes the *same* RSLC pair and lets you
override any existing runconfig key, crop to an AOI, and cache inputs for fast reruns.

| Knob | Override key (`--override` / `overrides=`) | Effect |
|---|---|---|
| Multilook | `processing.crossmul.range_looks` / `azimuth_looks` | resolution ↔ speckle |
| Geocode posting | `processing.geocode.output_posting.A.x_posting` / `y_posting` | output pixel spacing |
| Goldstein filter | `processing.filter_interferogram.*` | fringe smoothing over vegetation |
| Unwrap looks | `processing.phase_unwrap.range_looks` / `azimuth_looks` | unwrap robustness |
| Unwrap **bridge** | `processing.phase_unwrap.bridge.enabled` | stitch components across gaps |
| Ionosphere | `processing.ionosphere_phase_correction.enabled` | split-spectrum iono (demo: **off**) |
| Troposphere | `processing.troposphere_delay.enabled` | ECMWF tropo (demo: **off**) |
| Coreg refinement | `processing.dense_offsets` / `rubbersheet` / `fine_resample` `.enabled` | best **speed** cut (~22%, Δcoh 0.003) |
| Crop | `subset=[lon_min,lat_min,lon_max,lat_max]` | process a small patch |
| Cache | `cache_dir=...` | reuse cropped RSLCs + ancillaries for fast reruns |

**How you invoke it** (CLI or Python — both call `process_isce3`):

In [ ]:
# The custom run below was produced with this call (crossmul 2x2, geocode 20 m; vs standard 5x6, 80 m).
# It reuses the cached Cerro Prieto crop, so it finishes in ~50 s instead of a ~6 min cold run.
#
#   python -m hyp3_isce3 <ref_rslc> <sec_rslc> \
#       --subset "-115.30 32.34 -115.18 32.46" \
#       --cache-dir {BASE}/cerro_cache \
#       --override '{"processing.crossmul.range_looks": 2,
#                    "processing.crossmul.azimuth_looks": 2,
#                    "processing.geocode.output_posting.A.x_posting": 20,
#                    "processing.geocode.output_posting.A.y_posting": 20}'
#
# Equivalent Python (uncomment to run live ~50 s):
# from hyp3_isce3.process import process_isce3
# process_isce3(REF, SEC, subset=[-115.30,32.34,-115.18,32.46],
#               cache_dir=f'{BASE}/cerro_cache',
#               overrides={'processing.crossmul.range_looks':2,'processing.crossmul.azimuth_looks':2,
#                          'processing.geocode.output_posting.A.x_posting':20,
#                          'processing.geocode.output_posting.A.y_posting':20})
print('Cerro Prieto pair: 2025-12-03 -> 2025-12-27 (24-day, geothermal, MX)')

In [ ]:
# Standard-parameter run (5x6 looks, 80 m -- reproduces the operational config) vs custom (2x2, 20 m).
STD = glob.glob(f'{BASE}/exp_baseline/NISAR_L2_OD_GUNW_*.h5')[0]
CUS = glob.glob(f'{BASE}/cerro_custom/NISAR_L2_OD_GUNW_*.h5')[0]
AOI = (-115.30, 32.34, -115.18, 32.46)

wstd, ext = read_gunw(STD, AOI, 'wrapped'); cstd, cext = read_gunw(STD, AOI, 'coh')
wcus, exc = read_gunw(CUS, AOI, 'wrapped'); ccus, ccx = read_gunw(CUS, AOI, 'coh')

fig, ax = plt.subplots(2, 2, figsize=(12, 10))
ax[0,0].imshow(wphase(wstd), cmap='hsv', extent=ext,  origin='upper', vmin=-np.pi, vmax=np.pi)
ax[0,0].set_title('STANDARD 5x6 looks — wrapped phase')
ax[0,1].imshow(cstd, cmap='gray', extent=cext, origin='upper', vmin=0, vmax=1)
ax[0,1].set_title(f'STANDARD coherence (mean {np.nanmean(cstd):.2f})')
ax[1,0].imshow(wphase(wcus), cmap='hsv', extent=exc,  origin='upper', vmin=-np.pi, vmax=np.pi)
ax[1,0].set_title('CUSTOM 2x2 looks, 20 m — wrapped phase')
ax[1,1].imshow(ccus, cmap='gray', extent=ccx, origin='upper', vmin=0, vmax=1)
ax[1,1].set_title(f'CUSTOM coherence (mean {np.nanmean(ccus):.2f})')
for a in ax.ravel(): a.set_xlabel('UTM E [km]')
plt.tight_layout(); plt.show()

**Takeaway.** Fewer looks + tighter posting gives **finer resolution but more speckle** — coherence
is essentially unchanged (multilooking is a resolution↔noise trade, not a coherence recovery). This is
the *mechanism* for the finer-multilook knob; it pays off only where the operational 80 m grid
under-samples dense **coherent** fringes (steep compact gradients), not over broad fringes or
decorrelated ground.

---
## 2. Mexico City — more pairs: cumulative sum **vs** MintPy

A single 12-day pair captures ~12 days of motion. To see the multi-cm/yr subsidence of Mexico City
you combine many pairs. The **naive** way is to sum consecutive unwrapped pairs; the **rigorous** way
is a small-baseline time-series inversion (**MintPy**), which uses a redundant pair network, a
reference point, and (optionally) tropospheric/ramp corrections to reduce error.

In [ ]:
# --- Naive cumulative sum of the operational 12-day GUNW chain over Mexico City ---
import asf_search as asf
from datetime import datetime
asf.constants.INTERNAL.CMR_TIMEOUT = 90

MC_AOI = (-99.20, 19.15, -98.85, 19.55)   # basin incl. airport + Chalco/Xochimilco
def box(lon, lat, d=0.05):
    return f'POLYGON(({lon-d} {lat-d},{lon+d} {lat-d},{lon+d} {lat+d},{lon-d} {lat+d},{lon-d} {lat-d}))'

r = asf.search(dataset=asf.DATASET.NISAR, processingLevel='GUNW',
               intersectsWith=box(-99.03, 19.35), maxResults=80)
chain = []
for p in r:
    pr = p.properties['sceneName'].split('_')
    if pr[5] == '113' and pr[7] == '079':                       # track 113, frame 079
        try:
            dt = abs((datetime.strptime(pr[13][:8], '%Y%m%d') - datetime.strptime(pr[11][:8], '%Y%m%d')).days)
        except Exception:
            continue
        if dt == 12:
            urls = [p.properties.get('url')] + (p.properties.get('additionalUrls') or [])
            hh = [u for u in urls if u and u.endswith('.h5') and 'QA' not in u]
            if hh:
                chain.append((pr[11][:8], pr[13][:8], hh[0]))
chain = sorted(set(chain))
print('12-day pairs found:', [(a, b) for a, b, _ in chain])

In [ ]:
cum = cnt = None
for d1, d2, url in chain:
    ph, ext = read_gunw(url, MC_AOI, 'phase', stream=True)
    los = to_los_cm(ph)
    if cum is None:
        cum = np.zeros_like(los); cnt = np.zeros(los.shape, int); H, W = los.shape
    los = los[:H, :W]; g = np.isfinite(los)
    cum[g] += los[g]; cnt += g
cumd = np.where(cnt >= max(2, len(chain) - 1), cum, np.nan)

plt.figure(figsize=(7, 6))
im = plt.imshow(cumd, cmap='RdBu', vmin=-8, vmax=8, extent=ext, origin='upper')
plt.colorbar(im, label='cumulative LOS [cm]')
plt.title(f'Naive cumulative sum of {len(chain)} operational 12-day pairs')
plt.xlabel('UTM E [km]'); plt.ylabel('UTM N [km]'); plt.show()
v = cumd[np.isfinite(cumd)]
print(f'cumulative p2p (5-95%) = {np.nanpercentile(v,95)-np.nanpercentile(v,5):.1f} cm')

**Why go to MintPy?** The naive sum has no reference point, propagates unwrap errors pair-to-pair,
and leaves residual atmosphere/ramps in. MintPy runs a proper SBAS inversion over a redundant network.

> **The cells below require MintPy, which is *not* installed in this env.** Run the install cell once,
> then the stack-prep + inversion cells. (Left as a scaffold so this notebook builds without the heavy dependency.)

In [ ]:
# --- INSTALL (run once) ---  MintPy is a large dependency; installs into the active env.
# !mamba install -y -c conda-forge mintpy

In [ ]:
# --- MintPy scaffold (needs MintPy + a local GUNW stack) ---
# MintPy ingests a stack of NISAR GUNWs, builds the pair network, and inverts a time series.
# Exact prep depends on your MintPy version's NISAR-GUNW reader; the flow is:
#
# 1) Stage the operational GUNW stack locally (or point at generated GUNWs):
#      for _,_,url in chain: earthaccess.download([url], f'{BASE}/mexico_gunw_stack')
#
# 2) Prepare MintPy inputs + config (smallbaselineApp.cfg), e.g.:
#      mintpy.load.processor      = nisar
#      mintpy.load.unwFile        = {BASE}/mexico_gunw_stack/*GUNW*.h5
#      mintpy.reference.lalo      = 19.30, -99.05     # stable reference point
#      mintpy.troposphericDelay.method = pyaps        # or 'no'
#
# 3) Run the inversion:
#      !smallbaselineApp.py smallbaselineApp.cfg
#
# 4) Load the result and compare to the naive cumulative sum above:
#      from mintpy.utils import readfile
#      ts, atr = readfile.read('timeseries_ERA5_demErr.h5')   # [n_dates, H, W]
#      vel, _  = readfile.read('velocity.h5')                 # LOS velocity [m/yr]
#      # plot vel side-by-side with cumd; MintPy removes ramps/atmo the naive sum leaves in.
print('MintPy scaffold — install MintPy and stage the GUNW stack to run this section.')

---
## 3. Wet snow — a temporal **skip pair** (bridging over the wet acquisition)

Dry snow is ~transparent at L-band, but **wet / isothermal** snow decorrelates. When one acquisition
in a 12-day pair catches wet snow, coherence collapses. If no operational skip pair exists, this repo
can generate one that **bridges over** the wet date with a longer temporal baseline — restoring
coherence. Susanville (NE California high desert) had ephemeral wet snow in early Jan 2026.

In [ ]:
# Wet 12-day operational pair (catches wet snow) vs the generated skip pair (bridges over it).
import asf_search as asf
from datetime import datetime
SUS_AOI = (-120.75, 40.35, -120.45, 40.62)

# find the wet 12-day operational GUNW (track 143, frame 068, 2025-12-26 -> 2026-01-07)
r = asf.search(dataset=asf.DATASET.NISAR, processingLevel='GUNW',
               intersectsWith='POINT(-120.60 40.48)', maxResults=80)
wet_url = None
for p in r:
    pr = p.properties['sceneName'].split('_')
    if pr[5] == '143' and pr[7] == '068' and pr[11][:8] == '20251226' and pr[13][:8] == '20260107':
        urls = [p.properties.get('url')] + (p.properties.get('additionalUrls') or [])
        wet_url = next(u for u in urls if u and u.endswith('.h5') and 'QA' not in u)
print('wet 12-day GUNW:', 'found' if wet_url else 'NOT FOUND')

SKIP = glob.glob(f'{BASE}/susan_skip/NISAR_L2_OD_GUNW_*.h5')[0]   # generated Dec14 -> Jan19 skip pair
print('skip GUNW (local):', SKIP.split('/')[-1])

In [ ]:
cwet, ew = read_gunw(wet_url, SUS_AOI, 'coh', stream=True)
cskip, es = read_gunw(SKIP, SUS_AOI, 'coh')

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
ax[0].imshow(cwet, cmap='gray', vmin=0, vmax=1, extent=ew, origin='upper')
ax[0].set_title(f'WET 12-day pair  Dec26->Jan07\ncoherence mean {np.nanmean(cwet):.2f}')
ax[1].imshow(cskip, cmap='gray', vmin=0, vmax=1, extent=es, origin='upper')
ax[1].set_title(f'SKIP pair  Dec14->Jan19 (bridges wet date)\ncoherence mean {np.nanmean(cskip):.2f}')
ax[2].hist(cwet[np.isfinite(cwet)].ravel(), bins=40, range=(0, 1), alpha=.6, label='wet 12-day', density=True)
ax[2].hist(cskip[np.isfinite(cskip)].ravel(), bins=40, range=(0, 1), alpha=.6, label='skip pair', density=True)
ax[2].axvline(0.4, color='k', ls='--', lw=1); ax[2].set_xlabel('coherence'); ax[2].set_title('coherence distribution'); ax[2].legend()
for a in ax[:2]: a.set_xlabel('UTM E [km]')
plt.tight_layout(); plt.show()
print(f'coherent fraction (>0.4): wet {np.mean(cwet[np.isfinite(cwet)]>0.4):.2f}  ->  skip {np.mean(cskip[np.isfinite(cskip)]>0.4):.2f}')

**Takeaway.** The wet-snow acquisition drags the 12-day pair into decorrelation; the **skip pair**
(longer temporal baseline over a coherent-before / coherent-after window) recovers usable coherence.
When the operational catalog has no such skip pair, this repo generates one — the temporal-baseline knob.

---
### Summary
| Knob | Demo | Verdict |
|---|---|---|
| custom multilook / posting | §1 Cerro Prieto | mechanism shown (resolution↔speckle) |
| more pairs (stack → MintPy) | §2 Mexico City | naive sum live; MintPy scaffolded |
| temporal skip pair | §3 Susanville wet snow | recovers coherence ✅ |
| speed cut: coreg refinement off | `dense_offsets/rubbersheet/fine_resample.enabled=false` | ~22% faster, Δcoh 0.003 |